# RF Propagation — Equations as Code

Companion to **EE 625 Radio Wave Propagation**. Each key equation is encoded as a small,
self-contained function (numpy + matplotlib only) with a quick demo. Ordered by the
basic to advanced pipeline in `README.md`.

| # | Equation | Chapter | Pipeline tier |
|---|----------|---------|---------------|
| 0 | Frequency bands / power density / radio horizon | 1.1-1.2 | 0 |
| 1 | Free-space path loss (FSPL) | 4.2 | 0 |
| 2 | Log-distance path loss | 9.3.4 | 0 |
| 3 | ITU indoor path loss | 9.3.3 | 0 |
| 4 | Material properties: permittivity, conductivity, index | 2.2-2.3 | 1 |
| 5 | Waves in matter: complex ε, attenuation α/β, skin depth, impedance | 2.4 | 1 |
| 6 | Fresnel reflection & transmission | 2.6 | 1 |
| 7 | Fresnel zone radius | 8.2.2 | 2 |
| 8 | Knife-edge diffraction loss | 8.2.4 | 2 |
| 9 | Antenna & Tx source: gain, effective area, far-field, PLF | 3 | 0-1 |
| 10 | Link budget: margin, Friis, noise floor kT₀B, noise figure | 4.1-4.3 | 0 |
| 11 | Detailed link budget (Fig 4.4), interference margin, Eb/N0 | 4.4-4.6 | 0 |
| 12 | Radar range equation & RCS (future radar/scattering mode) | 5 | future |
| 13 | Atmospheric refraction, radio horizon, gaseous absorption | 6 | future (outdoor) |
| 14 | Near-Earth: foliage, terrain, **urban macro-models (Hata/COST-231/Lee)** | 7 | outdoor track |
| 15 | Two-ray ground bounce, Rayleigh roughness, knife-edge (Lee) | 8.1-8.2 | 1-2 |
| 16 | Log-normal shadowing, Rayleigh/Ricean, delay & Doppler spread | 8.3-8.5 | 2-3 |
| 17 | Indoor: ITU & log-distance path loss, floor loss, impulse response | 9 | 0 (engine core) |
| 18 | Chapter 9 exercises — worked with the encoded functions | 9 ex | — |
| 19 | Rain & fog attenuation: ITU model, specific attenuation (future outdoor mmWave) | 10 | future (outdoor) |
| 20 | Satellite links: slant-range geometry, ITU sat rain, hot-pad noise | 11 | future (outdoor) |

*(From Ch 3 on, sections are appended in reading order, not strict pipeline order.
Ch 5-7, 10, 11 are outdoor/future-feature models, kept separate from the indoor core.)*

> Engine hook notes are in each section. Tiers 3 (eikonal, FDTD) are numerical solvers,
> not closed-form equations — they live in the engine, not this notebook.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 2.99792458e8        # speed of light, m/s
EPS0 = 8.8541878128e-12 # vacuum permittivity, F/m

def wavelength(f_hz):
    return C / f_hz


## 0. Foundations (Ch 1) — bands, spreading, horizon

Building blocks from Ch 1: the **band-letter lookup** (1.1), the inverse-square **power
spreading** that FSPL is built on (eq 1.1), and — outdoor only — the **radio horizon**
(eq 1.4). Wavelength `lambda = c/f` is the `wavelength()` helper defined in Setup.


In [ ]:
# --- Frequency bands (Ch 1.1, Tables 1.1 & 1.2) ---
IEEE_BANDS = [    # (name, abbr, f_lo_hz, f_hi_hz)
    ("Medium",    "MF",  300e3, 3e6),   ("High",       "HF",  3e6,   30e6),
    ("Very high", "VHF", 30e6,  300e6), ("Ultra-high", "UHF", 300e6, 3e9),
    ("Super-high","SHF", 3e9,   30e9),  ("Extra-high", "EHF", 30e9,  300e9),
]
LETTER_BANDS = [  # nominal microwave / radar letter bands
    ("L",1e9,2e9), ("S",2e9,4e9), ("C",4e9,8e9), ("X",8e9,12e9), ("Ku",12e9,18e9),
    ("K",18e9,27e9), ("Ka",27e9,40e9), ("V",40e9,75e9), ("W",75e9,110e9),
]
def band_of(f_hz):
    # IEEE band abbreviation + microwave letter band for a frequency.
    ieee = next((a for _, a, lo, hi in IEEE_BANDS if lo <= f_hz < hi), None)
    ltr  = next((n for n, lo, hi in LETTER_BANDS if lo <= f_hz < hi), None)
    return {"IEEE": ieee, "letter": ltr}

for f in (2.4e9, 5.5e9, 28e9):
    print(f"{f/1e9:5.1f} GHz -> {band_of(f)}")

# --- Power density: inverse-square spreading (Ch 1.2, eq 1.1) ---
def power_density(p_watts, d_m):
    # W/m^2 at distance d from an isotropic source. The 1/d^2 root of FSPL.
    return p_watts / (4*np.pi*np.asarray(d_m, float)**2)

print("S(1 W isotropic, 10 m) =", power_density(1.0, 10.0), "W/m^2")


In [ ]:
# --- Radio horizon (Ch 1.2.1, eq 1.4, 4/3-earth). OUTDOOR / LOS only. ---
def radio_horizon_mi(h_ft):
    # Distance to the radio horizon in miles for antenna height h in feet.
    return np.sqrt(2.0*np.asarray(h_ft, float))

# reproduce Example 1.1: a 100 ft tower and a 50 ft tower
d1, d2 = radio_horizon_mi(100), radio_horizon_mi(50)
print(f"radio horizon: 100ft={d1:.1f} mi, 50ft={d2:.1f} mi, max link={d1+d2:.1f} mi")


## 1. Free-Space Path Loss (FSPL) — Ch 4.2

$$\mathrm{FSPL(dB)} = 20\log_{10}\!\left(\frac{4\pi d f}{c}\right)
= 32.44 + 20\log_{10} d_{km} + 20\log_{10} f_{MHz}$$

**Engine hook:** the Tier-0 sanity floor. Any `PL(x,y)` cell must be ≥ FSPL at that range.


In [ ]:
def fspl_db(d_m, f_hz):
    # Free-space path loss in dB. d_m > 0.
    return 20*np.log10(4*np.pi*np.asarray(d_m, float)*f_hz / C)

def fspl_db_eng(d_km, f_mhz):
    # Engineering form; equivalent to fspl_db for the same distance/frequency.
    return 32.44 + 20*np.log10(d_km) + 20*np.log10(f_mhz)

# demo: 2.4 GHz over 1..100 m
f = 2.4e9
d = np.linspace(1, 100, 200)
plt.figure(figsize=(6,4))
plt.plot(d, fspl_db(d, f))
plt.xlabel("distance (m)"); plt.ylabel("FSPL (dB)")
plt.title(f"FSPL at {f/1e9:.1f} GHz"); plt.grid(True); plt.show()

# spot check equivalence of the two forms
print(fspl_db(1000, 2.4e9), fspl_db_eng(1.0, 2400.0))


## 2. Log-Distance Path Loss — Ch 9.3.4

$$PL(d) = PL(d_0) + 10\,n\log_{10}\!\frac{d}{d_0} + X_\sigma$$

`n` = path-loss exponent (2 = free space, 3–5 indoors), `X_σ` = log-normal shadowing.

**Engine hook:** cheapest `PL(x,y)` layer. `n` absorbs average clutter; add walls
explicitly via the ITU/Motley–Keenan term (next).


In [ ]:
def log_distance_pl_db(d_m, n=3.0, pl_d0_db=40.0, d0_m=1.0, sigma_db=0.0, rng=None):
    d_m = np.asarray(d_m, float)
    mean = pl_d0_db + 10.0*n*np.log10(d_m/d0_m)
    if sigma_db <= 0:
        return mean
    r = rng or np.random.default_rng(0)
    return mean + r.normal(0.0, sigma_db, size=np.shape(d_m))

d = np.linspace(1, 50, 200)
plt.figure(figsize=(6,4))
for n in (2.0, 3.0, 4.0):
    plt.plot(d, log_distance_pl_db(d, n=n), label=f"n={n}")
plt.xlabel("distance (m)"); plt.ylabel("PL (dB)")
plt.title("Log-distance path loss"); plt.legend(); plt.grid(True); plt.show()


## 3. ITU Indoor Path Loss — Ch 9.3.3

$$PL = 20\log_{10} f_{MHz} + N\log_{10} d + L_f(n) - 28$$

`N` = distance power-loss coefficient (e.g. ~28 office / ~30 residential @ ~2 GHz),
`L_f(n)` = floor-penetration loss for `n` floors.

**Engine hook:** `N` and `L_f` map directly onto the Motley–Keenan distance term and the
floor-count penalty already in the path-loss layer.


In [ ]:
def itu_indoor_pl_db(d_m, f_mhz, N=28.0, Lf=0.0):
    # ITU-R P.1238 form. Lf is total floor-penetration loss (dB) for the path.
    return 20*np.log10(f_mhz) + N*np.log10(np.asarray(d_m, float)) - 28.0 + Lf

d = np.linspace(1, 50, 200)
plt.figure(figsize=(6,4))
for N, lab in ((22,"open"), (28,"office"), (30,"residential")):
    plt.plot(d, itu_indoor_pl_db(d, 2400.0, N=N), label=f"N={N} ({lab})")
plt.xlabel("distance (m)"); plt.ylabel("PL (dB)")
plt.title("ITU indoor path loss @ 2.4 GHz"); plt.legend(); plt.grid(True); plt.show()


## 4. Material Properties: Permittivity, Conductivity, Index — Ch 2.2-2.3

The engine's whole material model reduces to two numbers per material — relative
permittivity `eps_r` and conductivity `sigma` — because the book assumes **mu_r = 1**
(non-magnetic, §2.3). That single assumption is why the refractive index is
`n = sqrt(eps_r)` and why the Fresnel coefficients (§6) take `eps_r` alone.

- `eps_r` (dielectric constant, Table 2.2) -> the *real* part; sets reflection / refraction.
- `sigma` S/m (Table 2.3) -> feeds the *imaginary* part (complex permittivity, §5); sets loss.
- Constitutive: `E = D/eps`, `B = mu*H`. Boundary rules: normal `D` continuous, tangential `E` continuous.


In [ ]:
MU0 = 4*np.pi*1e-7   # H/m, vacuum permeability (EPS0 is defined in Setup)

# Relative permittivity (Table 2.2) and conductivity S/m (Table 2.3). Source: Seybold/Plonus.
# A complete engine entry needs BOTH; these are the materials the book pairs up.
MATERIALS = {   # name: (eps_r, sigma_S_per_m)
    "vacuum":          (1.0,    0.0),
    "air":             (1.0006, 0.0),
    "rubber":          (3.0,    1e-15),
    "quartz":          (5.0,    1e-17),
    "lead_glass":      (6.0,    1e-12),   # eps_r: lead glass; sigma: generic glass ~1e-12
    "mica":            (6.0,    1e-15),
    "distilled_water": (81.0,   1e-4),
}
# eps_r only (book gives no sigma): polystyrene 2.7, bakelite 5
# sigma only (conductors/earth): silver 6.1e7, copper 5.7e7, aluminum 3.5e7, seawater 4,
#   wet_earth ~1e-3, dry_earth ~1e-5, rock ~1e-6   (metals ~1e7 => near-perfect reflectors)

def refractive_index(eps_r, mu_r=1.0):
    # n = sqrt(mu_r * eps_r). With the book's mu_r = 1, n = sqrt(eps_r).
    return np.sqrt(eps_r*mu_r)

for name, (er, sig) in MATERIALS.items():
    print(f"{name:16s} eps_r={er:<7g} sigma={sig:<9g} n={refractive_index(er):.3f}")


In [ ]:
# Static E-FIELD refraction at a dielectric boundary (Ch 2.2, eqs 2.1-2.2).
# WARNING: this bends the E-FIELD VECTOR, not the propagation ray. It is NOT Snell's law
# (the book footnotes the distinction). It even bends the OPPOSITE sense to a ray: the E
# field tilts AWAY from the normal into higher eps_r, whereas a ray bends TOWARD the normal.
# Wave/ray refraction (Snell) comes from the boundary physics of Ch 2.6 -> see Fresnel (§6).
def efield_refraction_angle(phi1_rad, eps_r2, eps_r1=1.0):
    # Angle of E from the boundary normal in medium 2, given the angle in medium 1.
    return np.arctan((eps_r2/eps_r1)*np.tan(phi1_rad))

phi1 = np.radians(np.linspace(0, 89, 200))
plt.figure(figsize=(6,4))
for er2 in (3, 6, 81):
    plt.plot(np.degrees(phi1), np.degrees(efield_refraction_angle(phi1, er2)), label=f"eps_r2={er2}")
plt.plot(np.degrees(phi1), np.degrees(phi1), "k--", lw=0.8, label="no bend")
plt.xlabel("E angle from normal, medium 1 (deg)")
plt.ylabel("E angle from normal, medium 2 (deg)")
plt.title("Static E-field refraction (eq 2.1) — NOT Snell"); plt.legend(); plt.grid(True); plt.show()

print("phi1=0 ->", np.degrees(efield_refraction_angle(0.0, 81)), "deg (unrefracted)")
print("phi1=45deg, eps_r2=81 ->", round(float(np.degrees(efield_refraction_angle(np.radians(45), 81))),2), "deg")


## 5. Waves in Matter: Complex ε, Attenuation, Skin Depth — Ch 2.4

$$\varepsilon_c = \varepsilon' - j\varepsilon'' = \varepsilon_r - j\frac{\sigma}{\omega\varepsilon_0},
\qquad \tan\delta = \frac{\sigma}{\omega\varepsilon_r\varepsilon_0}$$

**Engine hook:** the imaginary part is *absorption*. Feed `eps_c` straight into the
Fresnel coefficients (next) so lossy walls both reflect and attenuate.


In [ ]:
def complex_permittivity(eps_r, sigma, f_hz):
    # Relative complex permittivity eps_c (dimensionless).
    w = 2*np.pi*f_hz
    return eps_r - 1j*sigma/(w*EPS0)

def loss_tangent(eps_r, sigma, f_hz):
    return sigma/(2*np.pi*f_hz*eps_r*EPS0)

# concrete-ish wall @ 2.4 GHz: eps_r ~ 5.24, sigma ~ 0.15 S/m
print("eps_c =", complex_permittivity(5.24, 0.15, 2.4e9))
print("tan(delta) =", loss_tangent(5.24, 0.15, 2.4e9))


**§2.4 wave-in-matter quantities** — all from the same material constants:
phase velocity `v = c/√(εr·μr)` (eq 2.4) · intrinsic impedance `Z0 = 377·√(μr/εr) Ω` (eq 2.12) ·
attenuation `α` + phase `β` from the complex wave number (eqs 2.9a/2.9b) · good-conductor
skin depth `δ = 1/√(π f μ σ)` (eq 2.11). Loss regime by tanδ: `<0.1` low (dielectric-like),
`>10` high (conductor-like).


In [ ]:
def phase_velocity(eps_r, mu_r=1.0):
    return C/np.sqrt(eps_r*mu_r)                 # eq 2.4

def intrinsic_impedance(eps_r, mu_r=1.0):
    return 377.0*np.sqrt(mu_r/eps_r)             # eq 2.12

def loss_regime(eps_r, sigma, f_hz):
    lt = loss_tangent(eps_r, sigma, f_hz)
    tag = ("low-loss (dielectric)" if lt < 0.1 else
           "high-loss (conductor)" if lt > 10 else "intermediate")
    return tag, lt

for name, er, sig in [("lead_glass",6,1e-12), ("distilled_water",81,1e-4), ("aluminum~",1.0,3.5e7)]:
    tag, lt = loss_regime(er, sig, 2.4e9)
    print(f"{name:15s} v={phase_velocity(er)/1e8:.2f}e8 m/s  Z0={intrinsic_impedance(er):6.1f} ohm  tanδ={lt:.1e} -> {tag}")


In [ ]:
def wave_params_lossy(eps_r, sigma, f_hz, mu_r=1.0):
    # Attenuation alpha (Np/m) and phase beta (rad/m) in a lossy medium. Eqs 2.9a/2.9b.
    w = 2*np.pi*f_hz
    eps = eps_r*EPS0; mu = mu_r*MU0
    root = np.sqrt(1 + (sigma/(w*eps))**2)
    alpha = w*np.sqrt(mu*eps/2*(root - 1))
    beta  = w*np.sqrt(mu*eps/2*(root + 1))
    return alpha, beta

def skin_depth(sigma, f_hz, mu_r=1.0):
    # Good-conductor skin depth (m): delta = 1/sqrt(pi f mu sigma). Eq 2.11.
    return 1.0/np.sqrt(np.pi*f_hz*mu_r*MU0*sigma)

for f in (1e6, 2.4e9, 60e9):
    print(f"copper skin depth @ {f/1e9:8.4f} GHz = {skin_depth(5.7e7, f)*1e6:7.3f} um")
a, b = wave_params_lossy(5.24, 0.05, 2.4e9)
print(f"lossy wall (eps_r=5.24, sigma=0.05) @2.4GHz: alpha={a:.2f} Np/m = {8.686*a:.1f} dB/m")


## 6. Fresnel Reflection & Transmission — Ch 2.6

From medium 1 (default air) into medium 2 with relative permittivity `eps_r2`
(may be complex, from §5). With $n=\sqrt{\varepsilon_r}$ and
$\cos\theta_t=\sqrt{1-(n_1/n_2)^2\sin^2\theta_i}$:

$$\Gamma_{TE}=\frac{n_1\cos\theta_i-n_2\cos\theta_t}{n_1\cos\theta_i+n_2\cos\theta_t},
\qquad
\Gamma_{TM}=\frac{n_2\cos\theta_i-n_1\cos\theta_t}{n_2\cos\theta_i+n_1\cos\theta_t}$$

Power transmitted (lossless) $T = 1-|\Gamma|^2$. Brewster angle: $\Gamma_{TM}=0$.

**Engine hook:** `Gamma` is the per-surface reflection coefficient for the
reflection/refraction effects; `T` is the pass-through attenuation for absorption.


In [ ]:
def fresnel_coeffs(theta_i_rad, eps_r2, eps_r1=1.0):
    # Returns (Gamma_TE, Gamma_TM) reflection coefficients (complex).
    n1 = np.sqrt(np.asarray(eps_r1, complex))
    n2 = np.sqrt(np.asarray(eps_r2, complex))
    cos_i = np.cos(theta_i_rad)
    sin_t = (n1/n2)*np.sin(theta_i_rad)
    cos_t = np.sqrt(1 - sin_t**2)
    g_te = (n1*cos_i - n2*cos_t)/(n1*cos_i + n2*cos_t)
    g_tm = (n2*cos_i - n1*cos_t)/(n2*cos_i + n1*cos_t)
    return g_te, g_tm

theta = np.radians(np.linspace(0, 89.9, 400))
g_te, g_tm = fresnel_coeffs(theta, eps_r2=5.24)   # lossless concrete-ish
brewster = np.degrees(np.arctan(np.sqrt(5.24)))

plt.figure(figsize=(6,4))
plt.plot(np.degrees(theta), np.abs(g_te), label="|Γ_TE| (perp)")
plt.plot(np.degrees(theta), np.abs(g_tm), label="|Γ_TM| (parallel)")
plt.axvline(brewster, ls="--", c="gray", label=f"Brewster ≈ {brewster:.1f}°")
plt.xlabel("incidence angle (deg)"); plt.ylabel("|Γ|")
plt.title("Fresnel reflection, εr=5.24"); plt.legend(); plt.grid(True); plt.show()

# lossy wall: reflection stays high, transmitted power carries attenuation
eps_c = complex_permittivity(5.24, 0.15, 2.4e9)
g_te_l, _ = fresnel_coeffs(np.radians(30), eps_c)
print("|Γ_TE|@30° lossy =", abs(g_te_l), "  T ≈", 1 - abs(g_te_l)**2)


### §2.6 reconciliation — grazing vs normal, and the Brewster cross-check

The book derives Γ via a **transmission-line analogy** (effective wave impedances Z_L, Z_z1)
using the **grazing angle** (measured from the surface). `fresnel_coeffs()` above is the
equivalent standard-optics form measured **from the normal** (grazing = 90° − normal), so the
book's Fig 2.7 x-axis is the mirror of ours. Same physics — checks below:
- the book's critical/polarizing angle (eq 2.23, grazing) and our Brewster (arctan√εr, from
  normal) are **complementary** (sum to 90°);
- at **grazing** incidence Γ → −1 for **both** polarizations (the two-ray ground null);
- perfect-conductor limit |Γ| → 1.

Field transmission `τ = 1+Γ` (can exceed 1) vs power transmittance `T = 1−|Γ|²` (≤1) are
different quantities — don't mix them.


In [ ]:
def brewster_grazing_deg(eps_r2, eps_r1=1.0):
    return np.degrees(np.arcsin(np.sqrt(eps_r1/(eps_r1+eps_r2))))   # book eq 2.23 (from surface)

def brewster_normal_deg(eps_r2, eps_r1=1.0):
    return np.degrees(np.arctan(np.sqrt(eps_r2/eps_r1)))            # optics (from normal)

er = 5.24
g, nrm = brewster_grazing_deg(er), brewster_normal_deg(er)
print(f"Brewster: grazing(eq 2.23)={g:.1f} deg + normal(arctan√εr)={nrm:.1f} deg = {g+nrm:.1f}")

gte, gtm = fresnel_coeffs(np.radians(89.9), er)          # near grazing
print(f"near-grazing Γ: TE={gte.real:.3f}, TM={gtm.real:.3f}  (both -> -1)")

gte_c, gtm_c = fresnel_coeffs(np.radians(45), complex_permittivity(1.0, 1e7, 2.4e9))
print(f"conductor |Γ| @45deg: TE={abs(gte_c):.3f}, TM={abs(gtm_c):.3f}")


## 7. Fresnel Zone Radius — Ch 8.2.2

$$r_n = \sqrt{\frac{n\,\lambda\,d_1 d_2}{d_1 + d_2}}$$

**Engine hook:** clearance test — if an obstacle intrudes past ~0.6·r₁ of the first zone,
the path is effectively obstructed and the diffraction model (next) kicks in.


In [ ]:
def fresnel_zone_radius(n, wavelength_m, d1_m, d2_m):
    d1_m = np.asarray(d1_m, float); d2_m = np.asarray(d2_m, float)
    return np.sqrt(n*wavelength_m*d1_m*d2_m/(d1_m + d2_m))

lam = wavelength(2.4e9)
D = 20.0
d1 = np.linspace(0.1, D-0.1, 200)
r1 = fresnel_zone_radius(1, lam, d1, D-d1)
plt.figure(figsize=(6,3.5))
plt.plot(d1, r1); plt.plot(d1, -r1, color=plt.gca().lines[-1].get_color())
plt.xlabel("position along path (m)"); plt.ylabel("1st Fresnel radius (m)")
plt.title(f"1st Fresnel zone, {D} m link @ 2.4 GHz"); plt.grid(True); plt.show()
print("max r1 =", r1.max(), "m at mid-path")


## 8. Knife-Edge Diffraction Loss — Ch 8.2.4

Fresnel–Kirchhoff parameter and the ITU single-edge loss approximation:

$$v = h\sqrt{\frac{2}{\lambda}\left(\frac1{d_1}+\frac1{d_2}\right)},\qquad
J(v)=6.9+20\log_{10}\!\left(\sqrt{(v-0.1)^2+1}+v-0.1\right)\ \text{dB},\ v>-0.78$$

`h` = obstruction height above the LOS line (negative if the edge is below LOS).

**Engine hook:** this is the diffraction effect — the extra loss that lets signal bend
into the geometric shadow behind corners/edges instead of leaving it black.


In [ ]:
def knife_edge_v(h_m, wavelength_m, d1_m, d2_m):
    return np.asarray(h_m, float)*np.sqrt(2.0/wavelength_m*(1.0/d1_m + 1.0/d2_m))

def knife_edge_loss_db(v):
    v = np.asarray(v, float)
    J = 6.9 + 20*np.log10(np.sqrt((v-0.1)**2 + 1) + v - 0.1)
    return np.where(v > -0.78, J, 0.0)

v = np.linspace(-3, 5, 400)
plt.figure(figsize=(6,4))
plt.plot(v, knife_edge_loss_db(v))
plt.axvline(0, ls="--", c="gray", label="grazing (v=0, ~6 dB)")
plt.xlabel("Fresnel–Kirchhoff v"); plt.ylabel("diffraction loss (dB)")
plt.title("Knife-edge diffraction (ITU approx)"); plt.legend(); plt.grid(True)
plt.gca().invert_yaxis(); plt.show()

# worked example: 0.5 m intrusion, mid-path of a 20 m link @ 2.4 GHz
lam = wavelength(2.4e9)
vv = knife_edge_v(0.5, lam, 10.0, 10.0)
print("v =", vv, "  loss =", float(knife_edge_loss_db(vv)), "dB")


## 9. Antenna & Tx Source Model — Ch 3

The transmitter side of the link: how much power radiates in which direction (gain, effective
area, EIRP), where the ray / plane-wave model is valid (far-field), and what polarization
mismatch costs (PLF). Pipeline Tier 0–1 — feeds FSPL / Friis (§1). Triage: §3.2, §3.3, §3.5
are the engine-relevant parts; §3.4 antenna zoo + §3.6 pointing loss are reference/skim.

- Isotropic power density `S = P/(4πd²)` (eq 3.1) = `power_density()` from §0 — the gain reference.
- Gain `G = η·D` (dBi); aperture `G = 4π·Ae/λ²` (eq 3.3), `Ae = η·Ap` (eq 3.2); beamwidth
  rule `G ≈ 26000/(θ_az·θ_el)`. **`Ae = Gλ²/4π` is also the Friis receive aperture** — the
  piece that turns §1's FSPL into a real link.
- **Far-field** `d > 2D²/λ` (eq 3.9): pattern fully formed, gain angle-only, wavefront planar
  ⇒ the **ray-theory validity boundary** (ties to Ch 2). Reactive near-field `r < λ/2π` (eq 3.11).
- Impedance match: `ρ=(Z1−Z0)/(Z1+Z0)`, mismatch loss `1−ρ²`, `VSWR=(1+ρ)/(1−ρ)`.
- Polarization loss `F = cos²τ` (linear, eq for §3.5.2) / full elliptical (eq 3.13); circular↔linear ≈ 3 dB.


In [ ]:
def antenna_gain_aperture(Ae_m2, wavelength_m):
    return 4*np.pi*Ae_m2/wavelength_m**2            # eq 3.3 (linear ratio)

def effective_area(gain_linear, wavelength_m):
    return gain_linear*wavelength_m**2/(4*np.pi)    # inverse of 3.3; also the Friis Rx aperture

def effective_area_physical(Ap_m2, eta=0.6):
    return eta*Ap_m2                                # eq 3.2

def gain_from_beamwidth(az_deg, el_deg):
    return 26000.0/(az_deg*el_deg)                  # rule of thumb (linear ratio)

# Example 3.1: 30 cm circular aperture @ 39 GHz
Ap = np.pi*(0.15)**2
Ae = effective_area_physical(Ap, 0.6)
G  = antenna_gain_aperture(Ae, wavelength(39e9))
print(f"Ex 3.1: Ae={Ae:.4f} m^2, G={G:.0f} = {10*np.log10(G):.1f} dBi  (book 39.5 dBi)")
print(f"round-trip effective_area(G): {effective_area(G, wavelength(39e9)):.4f} m^2 (== Ae)")


In [ ]:
def far_field_distance(D_m, wavelength_m):
    return 2*D_m**2/wavelength_m                    # eq 3.9 (Fraunhofer)

def reactive_nearfield_radius(wavelength_m):
    return wavelength_m/(2*np.pi)                   # eq 3.11 (book's convention)
# regions:  reactive NF  r < lambda/2pi  <  radiating NF  <  2D^2/lambda  <  far field

# Example 3.4: 140 MHz monopole
lam3 = wavelength(140e6)
print(f"Ex 3.4: lambda={lam3:.3f} m, reactive near-field r < {reactive_nearfield_radius(lam3):.3f} m (book 0.341 m)")
# a 0.3 m antenna at 2.4 GHz: ray model valid beyond the far-field distance
print(f"0.3 m antenna @2.4 GHz: far-field d > {far_field_distance(0.3, wavelength(2.4e9)):.2f} m")


In [ ]:
def reflection_coeff_z(Z1, Z0=50.0):
    return abs((Z1 - Z0)/(Z1 + Z0))                 # eq 3.6 (magnitude)

def mismatch_loss(Z1, Z0=50.0):
    return 1 - reflection_coeff_z(Z1, Z0)**2        # eq 3.7 (fraction of power delivered)

def vswr(Z1, Z0=50.0):
    r = reflection_coeff_z(Z1, Z0)
    return (1 + r)/(1 - r)                          # eq 3.8

# Example 3.3: 50-ohm source driving a 73-ohm half-wave dipole
print(f"Ex 3.3: rho={reflection_coeff_z(73):.3f}, VSWR={vswr(73):.2f}, loss={10*np.log10(mismatch_loss(73)):.2f} dB")
print("   (book prints 0.23 / 1.6 / -0.24 dB -- those fit ~80 ohm, not 73; minor slip in the book example)")


In [ ]:
def plf_linear(tau_deg):
    return np.cos(np.radians(tau_deg))**2           # F = cos^2(tau)  (§3.5.2)

def xpd_linear(tau_deg):
    return np.sin(np.radians(tau_deg))**2           # XPD = sin^2(tau) = 1 - F

def plf_elliptical(AR_w_dB, AR_r_dB, dtau_deg):
    # Full polarization loss factor, eq 3.13. AR in dB; dtau = tilt-angle difference (deg).
    ARw = 10**(AR_w_dB/20.0); ARr = 10**(AR_r_dB/20.0)
    num = ((1+ARw**2)*(1+ARr**2) + 4*ARw*ARr
           + (1-ARw**2)*(1-ARr**2)*np.cos(np.radians(2*dtau_deg)))
    return num/(2*(1+ARw**2)*(1+ARr**2))

print(f"PLF linear: tau=0 -> {plf_linear(0):.2f}, 45 -> {plf_linear(45):.2f}, 90 -> {plf_linear(90):.2f}")
print(f"circular(0 dB) <-> linear(~inf) PLF = {10*np.log10(plf_elliptical(0, 60, 90)):.2f} dB (expect -3)")
# Example 3.5: tx RHCP AR=2 dB, rx AR=3 dB
print(f"Ex 3.5: worst(tau=90)={10*np.log10(plf_elliptical(2,3,90)):.2f} dB, "
      f"best(tau=0)={10*np.log10(plf_elliptical(2,3,0)):.2f} dB (book -0.35 / -0.01)")


## 10. Link Budget & Noise Floor — Ch 4.1–4.3

The "so what" layer: turn the engine's `PL(x,y)` into a coverage map. Link margin ties
together EIRP (Ch 3), path loss (the engine), Rx gain (Ch 3), and the receiver threshold —
which is set by the noise floor (§4.3). **Coverage = { margin(x,y) > 0 }.** Pipeline Tier 0.

- **Link margin** `M = EIRP − L_path + G_Rx − TH_Rx` (all dB).
- **Friis** `L = G_T G_R (λ/4πd)²` (eq 4.1); FSL without gains = `20log(4πd/λ)` = §1 `fspl_db`.
  The λ-term is only there because Friis uses Rx *gain* instead of effective area `Ae=Gλ²/4π` (Ch 3).
- **Thermal noise** `N = kT₀B` (eq 4.4) → floor `−174 dBm/Hz + 10log B + NF`; `NF = 1 + Te/T₀`.
- **Cascade** `F_tot = F₁ + (F₂−1)/G₁ + (F₃−1)/(G₁G₂) + …` — first stage (LNA) sets the floor;
  passive loss *before* it adds dB-for-dB. *(The book's printed eq 4.14 drops the −1, but its
  Example 4.3 answer only works with the −1 — so this uses the standard, example-consistent form.)*

§4.3 (noise) is receiver-system, not propagation — the engine doesn't compute it, but it sets
`TH_Rx`, the coverage cutoff the map is thresholded against.


In [ ]:
def link_margin_db(eirp_dbm, path_loss_db, g_rx_db, th_rx_dbm):
    # Master link-budget equation (all dB). Margin > 0 => the link closes.
    return eirp_dbm - path_loss_db + g_rx_db - th_rx_dbm

def friis_received_power_dbm(pt_dbm, gt_db, gr_db, d_m, f_hz):
    # Pr = Pt + Gt + Gr - FSPL (dB form of Friis, eq 4.1).
    return pt_dbm + gt_db + gr_db - fspl_db(d_m, f_hz)

# Example 4.1: 100 m, 10 GHz, Pt=0.1 W (20 dBm), Gt=Gr=5 dB, TH=-85 dBm
eirp = 20 + 5                       # dBm (Pt + Gt)
pl   = fspl_db(100, 10e9)
print(f"Ex 4.1: EIRP={eirp} dBm, FSL={pl:.1f} dB, margin={link_margin_db(eirp, pl, 5, -85):.1f} dB (book 22.6 dB)")


In [ ]:
def thermal_noise_dbm(B_hz, nf_db=0.0):
    # AWGN floor: -174 dBm/Hz (kT0 at 290 K) + 10log10(B) + noise figure.
    return -174.0 + 10*np.log10(B_hz) + nf_db

def noise_figure_db_from_temp(Te_K, T0=290.0):
    return 10*np.log10(1 + Te_K/T0)                 # F = 1 + Te/T0 (eq 4.12), in dB

def cascade_noise_factor(F_list, G_list):
    # Friis cascade, linear noise factors F and gains G (standard -1 form; see note above).
    F_tot = F_list[0]; g = 1.0
    for i in range(1, len(F_list)):
        g *= G_list[i-1]
        F_tot += (F_list[i] - 1.0)/g
    return F_tot

# Example 4.2: 10 Msym/s -> B ~ 10 MHz, Te=870 K
nf = noise_figure_db_from_temp(870)
N  = thermal_noise_dbm(10e6, nf)
print(f"Ex 4.2: NF={nf:.1f} dB, N={N:.0f} dBm = {N-30:.0f} dBW (book 6 dB, -128 dBW)")

# Example 4.3: 7 dB cable loss before a receiver of Te=630 K
nf_rx = noise_figure_db_from_temp(630)
print(f"Ex 4.3: Rx NF={nf_rx:.1f} dB + 7 dB cable = {nf_rx+7:.1f} dB (simple add)")
F_cable = 10**(7/10.0); G_cable = 1/F_cable         # passive attenuator: F = L, G = 1/L
F_tot = cascade_noise_factor([F_cable, 10**(nf_rx/10.0)], [G_cable, 1.0])
print(f"        cascade check: {10*np.log10(F_tot):.1f} dB")


In [ ]:
# Synthesis: noise floor -> threshold -> path-loss budget the engine's PL(x,y) is compared to
B, NF, snr_req = 20e6, 6.0, 10.0                    # 20 MHz, 6 dB NF, need 10 dB SNR
th = thermal_noise_dbm(B, NF) + snr_req             # receiver threshold, dBm
print(f"floor {thermal_noise_dbm(B,NF):.1f} dBm + SNR {snr_req:.0f} dB  ->  TH_Rx = {th:.1f} dBm")
eirp, gr = 25.0, 5.0
print(f"budget: EIRP {eirp:.0f} + Gr {gr:.0f} - TH {th:.1f} = {eirp+gr-th:.1f} dB path loss allowed")
print("=> coverage(x,y) = PL(x,y) < this  (equivalently link_margin > 0)")


## 11. Detailed Link Budget, Interference & Eb/N0 — Ch 4.4–4.6

The full itemized budget (book Fig 4.4) + interference margin + the SNR→Eb/N0 step. Fig 4.4 **is**
the engine's per-cell coverage template: at each (x,y), RSL = EIRP − PL(x,y) + Rx_gain, then
net margin = RSL − interference_margin − TH. The engine supplies PL(x,y); everything else is scalar.

- **EIRP** = P_Tx + G_Tx − L_WG − L_radome; **Rx gain** = G_Rx − L_radome − L_WG − L_pol − L_pt.
- **Interference margin** (Ex 4.4): a 1-dB margin ⇒ total interference must stay ≥ 5.9 dB below the
  noise floor. → `interference_for_margin_dbm()`.
- **Eb/N0 = SNR + 10log(B/Rb)** (use the *data* bit rate Rb). → `eb_n0_db()`.
- ⚠ The book's inline 4.5.2–4.5.5 numbers (PL 135 dB, EIRP 27 dBm, SNR 16 dB, 10log B = 60) are
  mutually inconsistent and mismatch Fig 4.4; the self-consistent Fig 4.4 set is used here.


In [ ]:
def interference_for_margin_dbm(noise_dbm, margin_db):
    # Max total interference power (dBm) for a given interference-margin (noise-floor rise).
    return noise_dbm + 10*np.log10(10**(margin_db/10.0) - 1)

def interference_margin_db(noise_dbm, interference_dbm):
    return 10*np.log10(1 + 10**((interference_dbm - noise_dbm)/10.0))

def eb_n0_db(snr_db, B_hz, Rb_bps):
    # Eb/N0 (dB) = SNR + 10log10(B/Rb). Use the DATA bit rate Rb, not the channel rate.
    return snr_db + 10*np.log10(B_hz/Rb_bps)

# Example 4.4: a 1-dB interference margin
print(f"Ex 4.4: 1-dB margin -> interference must stay {-interference_for_margin_dbm(0,1.0):.1f} dB below noise (book 5.9)")


In [ ]:
def link_budget(f_hz, d_m, tx_pwr_dbm, tx_gain_db, tx_loss_db, tx_radome_db,
                pl_extra_db, rx_gain_db, rx_radome_db, rx_loss_db, rx_pol_db, rx_pt_db,
                nf_db, bw_hz, snr_req_db, interf_margin_db=0.0):
    eirp = tx_pwr_dbm + tx_gain_db - tx_loss_db - tx_radome_db
    fsl = fspl_db(d_m, f_hz)
    total_pl = fsl + pl_extra_db
    rx_gain = rx_gain_db - rx_radome_db - rx_loss_db - rx_pol_db - rx_pt_db
    rsl = eirp - total_pl + rx_gain
    noise = thermal_noise_dbm(bw_hz, nf_db)
    snr = rsl - noise - interf_margin_db
    threshold = noise + snr_req_db
    net_margin = rsl - interf_margin_db - threshold
    return dict(EIRP=eirp, FSL=fsl, total_PL=total_pl, Rx_gain=rx_gain, RSL=rsl,
                noise=noise, SNR=snr, threshold=threshold, net_margin=net_margin)

# Reproduce book Figure 4.4: 38.6 GHz, 2 km terrestrial mmwave link
lb = link_budget(38.6e9, 2000, tx_pwr_dbm=10, tx_gain_db=32, tx_loss_db=1.5, tx_radome_db=2.0,
                 pl_extra_db=1.0+15.0+2.0+0.2,     # pointing + rain(0.999) + multipath + atmos
                 rx_gain_db=32, rx_radome_db=2.0, rx_loss_db=2.0, rx_pol_db=0.2, rx_pt_db=1.0,
                 nf_db=7.0, bw_hz=25e6, snr_req_db=5.0, interf_margin_db=1.0)
for k in ("EIRP","FSL","total_PL","Rx_gain","RSL","noise","SNR","threshold","net_margin"):
    print(f"  {k:11s} {lb[k]:7.1f}")
print("book Fig 4.4:  EIRP 38.5  FSL 130.2  PL 148.4  RxG 26.8  RSL -83.1  N -93.0  SNR 8.9  TH -88.0  margin 3.9")


## 12. Radar Range Equation & RCS — Ch 5 *(future feature)*

Off the indoor one-way path, but useful for a future **radar mode** or a **scattering
reflectivity** model. Key idea: a reflected return is *two-way* Friis → falls as **1/R⁴**
(doubling range = 12 dB, not 6). Reflectivity is **RCS** `σ` (m², or dBsm) — an *electrical*
area independent of physical size.

- `Pr = Pt Gt Gr λ² σ / ((4π)³ R⁴)` (eq 5.8) → `radar_rx_power_dbw()`.
- `Rmax = [Pt G² σ λ² / (Prmin (4π)³)]^(1/4)` (eq 5.9) → `radar_max_range_m()`.
- RCS shapes (Table 5.1): sphere `πr²`, flat plate `4π(lw)²/λ²`, trihedral `4πl⁴/(3λ²)`.
- **Clutter** (§5.4): area backscatter `σ⁰` (RCS/m²), volume `η` (RCS/m³). Clutter area grows
  with R so it falls off slower than a point target — the analog of diffuse wall scatter.


In [ ]:
def radar_rx_power_dbw(pt_dbw, gt_db, gr_db, rcs_m2, wavelength_m, R_m):
    return (pt_dbw + gt_db + gr_db + 10*np.log10(rcs_m2) + 20*np.log10(wavelength_m)
            - 30*np.log10(4*np.pi) - 40*np.log10(R_m))               # eq 5.8

def radar_max_range_m(pt_dbw, g_db, rcs_m2, wavelength_m, pr_min_dbw):
    num = pt_dbw + 2*g_db + 10*np.log10(rcs_m2) + 20*np.log10(wavelength_m)  # eq 5.9
    return 10**((num - (pr_min_dbw + 30*np.log10(4*np.pi)))/40)

def rcs_sphere(r_m):                 return np.pi*r_m**2
def rcs_flat_plate(l_m, w_m, wavelength_m): return 4*np.pi*(l_m*w_m)**2/wavelength_m**2

# Example 5.1: 2 GHz, Pt=0 dBW, G=18 dB, R=2 km, sigma=1 m2, B=50 kHz, F=5 dB
pr = radar_rx_power_dbw(0, 18, 18, 1.0, wavelength(2e9), 2000)
n  = -174 + 10*np.log10(50e3) + 5          # dBm
print(f"Ex 5.1: Pr={pr:.1f} dBW = {pr+30:.1f} dBm, N={n:.0f} dBm, SNR={pr+30-n:.1f} dB (book 6.5)")

# Example 5.2: 10 GHz, Pt=60 dBW, G=28 dB, tau=100us, sigma=1 m2 @ 20 km, Teff=200 K
pr2 = radar_rx_power_dbw(60, 28, 28, 1.0, wavelength(10e9), 20000)
n2  = -204 + 10*np.log10(1/100e-6) + 10*np.log10(1 + 200/290)   # dBW
print(f"Ex 5.2: Pr={pr2:.1f} dBW, N={n2:.1f} dBW, SNR={pr2-n2:.1f} dB (book 42.2)")


## 13. Atmospheric Refraction, Radio Horizon & Absorption — Ch 6 *(future, outdoor)*

Indoors these are negligible; they matter only for **long outdoor links** (outdoor city
track) and **mmWave** absorption. Extends §0's radio horizon with the real refraction model.

- Refractivity `N = (77.6/T)(P + 4810 e/T)` (eq 6.4); gradient `dN/dh = −(Ns/H) e^(−h/H)`,
  H=7 km. → `refractivity()`, `refractivity_gradient()`.
- Effective-earth factor `k = 1/(1 + r·dn/dh)` (eq 6.11); radio horizon `d ≈ √(2 k r h)`.
  → `k_factor()`, `radio_horizon_km()`. **Ducting** when `dn/dh = −157×10⁻⁶ /km` (k → ∞).
- Gaseous absorption `A = γ·d` (dB), `γ = γ_O2 + γ_H2O`. Lines: **22 GHz (water vapor),
  60 GHz (oxygen, ~15 dB/km)**; ~0.05 dB/km at 1 GHz. Radar → ×2. → `atmospheric_loss_db()`.


In [ ]:
def refractivity(P_mb, e_mb, T_K):
    return (77.6/T_K)*(P_mb + 4810*e_mb/T_K)                 # eq 6.4

def refractivity_gradient(Ns, h_km, H_km=7.0):
    return -Ns/H_km*np.exp(-h_km/H_km)                        # eq 6.7

def k_factor(dNdh_per_km, r_earth_km=6370.0):
    return 1.0/(1 + r_earth_km*dNdh_per_km*1e-6)              # eq 6.11

def radio_horizon_km(h_m, k=4/3, r_earth_km=6370.0):
    return np.sqrt(2*k*r_earth_km*h_m/1000.0)                 # d ~ sqrt(2 k r h)

def atmospheric_loss_db(gamma_db_per_km, d_km, radar=False):
    return gamma_db_per_km*d_km*(2 if radar else 1)           # A = gamma*d (x2 for radar)

# Example 6.1: 50 m tower at 2 km ASL; P=1100 mb, e=12 mb, T=260 K
Ns = refractivity(1100, 12, 260)
dNdh = refractivity_gradient(Ns, 2.0)
k = k_factor(dNdh)
print(f"Ex 6.1: N={Ns:.1f} (book 394.6), dN/dh={dNdh:.2f} (-42.36), k={k:.3f} (1.370), "
      f"horizon={radio_horizon_km(50, k):.1f} km (book 29.5)")


## 14. Near-Earth Models: Foliage, Terrain, Urban — Ch 7 *(outdoor city track)*

The empirical baseline + validation reference for the **outdoor OSM-voxelized sim**. All are
*median* path-loss fits (not physics). **Hata / COST-231 are the workhorses** for mobile urban.

- Foliage: Weissberger `1.33 F^0.284 d^0.588` (14–400 m) / `0.45 F^0.284 d` (<14 m), F in GHz
  (eq 7.1); early ITU `0.2 F^0.3 d^0.6`, F in MHz (eq 7.2). → `weissberger_foliage_db()`, `itu_foliage_db()`.
- Terrain: **Egli** 4th-power law `(hb hm/d²)²(40/f)²` (eq 7.9) → `egli_pl_db()`; **ITU terrain
  diffraction** `Ad = −20 h/F1 + 10` with `F1 = 17.3√(d1 d2/(f d))` (the Ch 8 Fresnel radius in
  km/GHz) → `itu_terrain_diffraction_db()`.
- **Urban macro-models:** **Hata** (150–1500 MHz, eq 7.14) urban/suburban/open → `hata_pl_db()`;
  **COST-231** (1500–2000 MHz PCS, eq 7.19) → `cost231_pl_db()`; **Lee** (fittable power law,
  eq 7.20) → `lee_pl_db()`. Okumura = the graphical parent of Hata; Young = NYC power law.


In [ ]:
def weissberger_foliage_db(d_f_m, f_ghz):
    if d_f_m <= 14: return 0.45*f_ghz**0.284*d_f_m               # 0 < d <= 14 m
    return 1.33*f_ghz**0.284*d_f_m**0.588                        # 14 < d <= 400 m

def itu_foliage_db(d_f_m, f_mhz):
    return 0.2*f_mhz**0.3*d_f_m**0.6                             # eq 7.2 (F in MHz)

def egli_pl_db(d_m, f_mhz, hb_m, hm_m, Gb=1.0, Gm=1.0):
    gain = Gb*Gm*(hb_m*hm_m/d_m**2)**2*(40.0/f_mhz)**2           # Pr/Pt (eq 7.9)
    return -10*np.log10(gain)

def fresnel_radius_terrain_m(d1_km, d2_km, d_km, f_ghz):
    return 17.3*np.sqrt(d1_km*d2_km/(f_ghz*d_km))               # eq 7.11 (= Ch 8 Fresnel radius)

def itu_terrain_diffraction_db(h_m, F1_m):
    return -20*(h_m/F1_m) + 10                                   # eq 7.10

# Example 7.1: 12 m of trees, 1 GHz
print(f"Ex 7.1: Weissberger={weissberger_foliage_db(12, 1.0):.2f} dB (book 5.4), "
      f"ITU={itu_foliage_db(12, 1000):.2f} dB (book 7.06)")
# Example 7.3: 3 km, 100 MHz, blockage mid-path, h=0.75 m
F1 = fresnel_radius_terrain_m(1.5, 1.5, 3.0, 0.1)
print(f"Ex 7.3: F1={F1:.1f} m (book 47.4), Ad={itu_terrain_diffraction_db(0.75, F1):.1f} dB (book 9.7)")
# Egli Example 7.2: 100 MHz, hb=20, hm=3, d=1 km
print(f"Ex 7.2: Egli={egli_pl_db(1000, 100, 20, 3):.1f} dB  "
      f"(book prints 112.4, but that used hb*hm=6 instead of 60; correct value is 92.4)")


In [ ]:
def hata_a_hr(hr_m, fc_mhz, large_city=False):
    if large_city:
        if fc_mhz <= 200: return 8.29*(np.log10(1.54*hr_m))**2 - 1.1
        return 3.2*(np.log10(11.75*hr_m))**2 - 4.97             # fc >= 400 MHz
    return (1.1*np.log10(fc_mhz) - 0.7)*hr_m - (1.56*np.log10(fc_mhz) - 0.8)

def hata_pl_db(d_km, fc_mhz, ht_m, hr_m, area="urban", large_city=False):
    a = hata_a_hr(hr_m, fc_mhz, large_city)
    urban = (69.55 + 26.16*np.log10(fc_mhz) - 13.82*np.log10(ht_m) - a
             + (44.9 - 6.55*np.log10(ht_m))*np.log10(d_km))       # eq 7.14
    if area == "suburban": return urban - 2*(np.log10(fc_mhz/28.0))**2 - 5.4      # eq 7.17
    if area == "open":     return urban - 4.78*(np.log10(fc_mhz))**2 + 18.33*np.log10(fc_mhz) - 40.94  # eq 7.18
    return urban

def cost231_pl_db(d_km, fc_mhz, ht_m, hr_m, metro=False, large_city=False):
    a = hata_a_hr(hr_m, fc_mhz, large_city)
    return (46.3 + 33.9*np.log10(fc_mhz) - 13.82*np.log10(ht_m) - a
            + (44.9 - 6.55*np.log10(ht_m))*np.log10(d_km) + (3.0 if metro else 0.0))  # eq 7.19

LEE_PARAMS = {   # environment: (L0_dB at 1 km, gamma dB/decade)  -- Table 7.2
    "free_space": (85, 20), "open_rural": (89, 43.5), "suburban": (101.7, 38.5),
    "philadelphia": (110, 36.8), "newark": (104, 43.1), "tokyo": (124.0, 30.5),
}
def lee_pl_db(d_km, L0_db, gamma, F0_db=0.0):
    return L0_db + gamma*np.log10(d_km) - F0_db                   # eq 7.20 (F0 already in dB)

# Example 7.5: ht=68, fc=870 MHz, hr=3, d=3.7 km, large city
print(f"Ex 7.5: Hata urban(large city) = {hata_pl_db(3.7, 870, 68, 3, large_city=True):.1f} dB (book 137.1)")
print(f"        a(hr) = {hata_a_hr(3, 870, True):.2f} (book 2.69)")
# COST-231 sanity (same geometry, PCS band 1800 MHz)
print(f"COST-231 @1800 MHz (metro) = {cost231_pl_db(3.7, 1800, 68, 3, metro=True, large_city=True):.1f} dB")
# Example 7.6 (Lee, suburban): L0=101.7, gamma=38.5, F0=-5 dB -> 106.7 + 38.5 log d
print(f"Ex 7.6: Lee suburban = {lee_pl_db(1.0, 101.7, 38.5, -5.0):.1f} + 38.5*log(d)  (book 106.7 + 38.5 log d)")


## 15. Ground-Bounce Multipath, Roughness & Diffraction — Ch 8.2

Core physics for the **two-ray model** (Tier 1) and the **diffraction effect**. Extends the
pre-seeded §7 (Fresnel zones) and §8 (knife-edge) — both **verified below** against the book's
Examples 8.2 & 8.3 (seeded `fresnel_zone_radius` = eq 8.20, `knife_edge_v` = eq 8.19: exact match).

- **Two-ray ground bounce:** ρ ≈ −1 at small grazing angle ⇒ over flat earth received power falls
  as **1/d⁴** (not 1/d²), *independent of λ* (eq 8.13). Crosses over from FSL at `dx = 4π ht hr/λ`
  (eq 8.15); exact field oscillates as `2·sin(Δθ/2)` (nulls & the 6 dB constructive peak).
- **Rayleigh roughness** `HR = λ/(8 sinθ)` (eq 8.16): Δh < HR ⇒ smooth/specular (**reflection**);
  Δh ≫ HR ⇒ rough/diffuse (**scattering**). The reflection-vs-scattering trigger the engine needs.
- **Knife-edge** via Lee's piecewise approx (eqs 8.21) — matches the book; §8's ITU J(v) is an
  equivalent alternative. v = 0 ⇒ 6 dB (50 % blocked); v = −0.8 (60 % clearance) ⇒ 0 dB. Odd
  Fresnel-zone boundaries ⇒ destructive interference.


In [ ]:
def two_ray_reflection_point(ht, hr, d):
    d1 = d*ht/(hr + ht); return d1, d - d1                     # specular point (phi1 = phi2)

def two_ray_phase_diff(ht, hr, d, wavelength):                 # eq 8.8 (exact)
    return (2*np.pi/wavelength)*d*(np.sqrt(1+((hr+ht)/d)**2) - np.sqrt(1+((ht-hr)/d)**2))

def two_ray_crossover_m(ht, hr, wavelength):
    return 4*np.pi*ht*hr/wavelength                            # eq 8.15

def two_ray_pathloss_db(ht, hr, d, wavelength, gt=1.0, gr=1.0, exact=False):
    g_fsl = gt*gr*(wavelength/(4*np.pi*d))**2
    if exact:                                                  # eq 8.9: Lmp = Lfsl * 4 sin^2(dtheta/2)
        return -10*np.log10(g_fsl*4*np.sin(two_ray_phase_diff(ht,hr,d,wavelength)/2)**2)
    g_2ray = gt*gr*(ht*hr)**2/d**4                             # eq 8.13
    return -10*np.log10(min(g_2ray, g_fsl))                    # eq 8.14: use whichever gives greater loss

# Example 8.1: ht=hr=10 m, 2 GHz
lam2 = wavelength(2e9)
print(f"Ex 8.1: crossover dx = {two_ray_crossover_m(10,10,lam2):.0f} m")
print(f"  d=4 km  -> PL={two_ray_pathloss_db(10,10,4000,lam2):.1f} dB (< dx, so FSL; book 110.5)")
print(f"  d=40 km -> PL={two_ray_pathloss_db(10,10,40000,lam2):.1f} dB (> dx, so 1/d^4; book 144.1)")
print(f"  Fig 8.5 crossover (ht=100,hr=3,lam=0.3) = {two_ray_crossover_m(100,3,0.3)/1000:.1f} km (book 12.6)")


In [ ]:
def rayleigh_roughness_m(wavelength, grazing_rad):
    return wavelength/(8*np.sin(grazing_rad))                  # eq 8.16

def is_specular(delta_h_m, wavelength, grazing_rad):
    # True -> smooth/specular (reflection); False -> rough/diffuse (scattering).
    return delta_h_m < rayleigh_roughness_m(wavelength, grazing_rad)

g = np.radians(5)
hr_thr = rayleigh_roughness_m(wavelength(2.4e9), g)
print(f"Rayleigh HR @2.4 GHz, 5deg grazing = {hr_thr*100:.1f} cm  "
      f"(bumps below -> specular reflection, above -> diffuse scatter)")


In [ ]:
def knife_edge_loss_lee_db(v):
    # Lee's piecewise approximation to the diffraction integral (eqs 8.21). Loss in dB (>=0).
    v = np.asarray(v, float)
    with np.errstate(invalid="ignore", divide="ignore"):
        b1 = -20*np.log10(0.5 - 0.62*v)                        # -1 <= v <= 0
        b2 = -20*np.log10(0.5*np.exp(-0.95*v))                 #  0 <= v <= 1
        b3 = -20*np.log10(0.4 - np.sqrt(0.1184 - (0.38-0.1*v)**2))  # 1 <= v <= 2.4
        b4 = -20*np.log10(0.225/v)                             # v >= 2.4
    return np.select([v <= -1, v <= 0, v <= 1, v <= 2.4], [0.0, b1, b2, b3], default=b4)

# --- Verify the pre-seeded §7 & §8 functions against the book ---
# Example 8.2 (Fresnel zone): 1 km, 28 GHz, blockage 300 m from one end
r1 = fresnel_zone_radius(1, wavelength(28e9), 300, 700)
print(f"Ex 8.2: 1st Fresnel radius = {r1:.2f} m, 60% clearance = {0.6*r1:.2f} m (book ~0.9 m)")
# Example 8.3 (knife-edge): 150 MHz, edge 5 m below LOS, 200 m from one end
v83 = knife_edge_v(-5, wavelength(150e6), 200, 800)
print(f"Ex 8.3: v = {v83:.3f} (book -0.395),  Lee loss = {float(knife_edge_loss_lee_db(v83)):.1f} dB "
      f"(book 2.6),  ITU J(v) = {float(knife_edge_loss_db(v83)):.1f} dB")
print(f"        clearance check: v=-0.8 -> {float(knife_edge_loss_lee_db(-0.8)):.1f} dB, "
      f"v=0 -> {float(knife_edge_loss_lee_db(0.0)):.1f} dB")


## 16. Log-Normal Shadowing & Small-Scale Fading — Ch 8.3–8.5

The **statistical layer** on top of the deterministic median path loss. Two independent axes:
*large-scale* shadowing (log-normal, `X_σ`) and *small-scale* fading (Rayleigh/Ricean).

- **§8.3 Log-normal shadowing:** many diffraction/reflection losses multiply ⇒ add in dB ⇒
  (CLT) Gaussian-in-dB. Margin `L_S = z·σ_L` for a target coverage; `z = Φ⁻¹(coverage)`.
  Edge coverage `= Φ(M/σ_L)`. σ_L(Okumura fit) `= 0.65(log fc)² − 1.3 log fc + A` (A=5.2 urban /
  6.2 suburban). → `shadowing_margin_db()`, `edge_coverage_prob()`, `location_variability_okumura()`.
- **§8.4 Small-scale:** all-reflections ⇒ **Rayleigh**; a dominant/LOS path ⇒ **Ricean** (factor K).
  → `rayleigh_fade_prob()`, `ricean_fade_prob()`.
- **Delay spread** σ_τ ⇒ coherence BW `Bc ≈ 1/(5σ_τ)…1/(50σ_τ)` (eq 8.24); B<Bc = flat, else
  selective. **Doppler** `fm = Δv/λ` (eq 8.26) ⇒ coherence time `Tc ≈ 1/fm` (eq 8.25); T_sym<Tc = slow.
  → `coherence_bandwidth_hz()`, `doppler_shift_hz()`, `coherence_time_s()`.

*Engine:* layer `X_σ ~ N(0, σ_L)` onto any deterministic PL(x,y) for a coverage-*probability* map;
`log_distance_pl()` already carries σ. Delay-spread ties to the exponential impulse response of Ch 9.


In [ ]:
import math

def norm_cdf(x):                                   # Phi(x)
    return 0.5*(1 + np.vectorize(math.erf)(np.asarray(x, float)/math.sqrt(2)))

def norm_ppf(p):                                   # Phi^-1 (Acklam), |err| < 1.15e-9
    a=[-3.969683028665376e1,2.209460984245205e2,-2.759285104469687e2,1.383577518672690e2,-3.066479806614716e1,2.506628277459239e0]
    b=[-5.447609879822406e1,1.615858368580409e2,-1.556989798598866e2,6.680131188771972e1,-1.328068155288572e1]
    c=[-7.784894002430293e-3,-3.223964580411365e-1,-2.400758277161838e0,-2.549732539343734e0,4.374664141464968e0,2.938163982698783e0]
    d=[7.784695709041462e-3,3.224671290700398e-1,2.445134137142996e0,3.754408661907416e0]
    def one(p):
        if p < 0.02425:
            q=math.sqrt(-2*math.log(p));  return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
        if p > 1-0.02425:
            q=math.sqrt(-2*math.log(1-p));return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5])/((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
        q=p-0.5; r=q*q; return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q/(((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)
    return np.vectorize(one)(np.asarray(p, float))

def shadowing_margin_db(sigma_L, coverage):        # L_S = z * sigma_L
    return norm_ppf(coverage)*sigma_L

def edge_coverage_prob(margin_db, sigma_L):
    return norm_cdf(margin_db/sigma_L)

def location_variability_okumura(fc_mhz, area="urban"):
    A = 5.2 if area == "urban" else 6.2
    return 0.65*(np.log10(fc_mhz))**2 - 1.3*np.log10(fc_mhz) + A

# Example 8.5: 90% edge coverage
for s in (6, 8):
    print(f"Ex 8.5: sigma_L={s} dB -> z={norm_ppf(0.9):.2f}, shadowing margin L_S={shadowing_margin_db(s,0.9):.2f} dB "
          f"(book {7.7 if s==6 else 10.24})")


In [ ]:
def rayleigh_fade_prob(fade_db):
    # P(signal >= fade_db below the average), non-LOS multipath. Closed form (integrated Rayleigh pdf).
    return 1 - np.exp(-0.5*10**(-np.asarray(fade_db, float)/10))

def ricean_fade_prob(fade_db, K_dB, n=40000):
    # P(envelope power >= fade_db below the diffuse scale sigma^2) for a Ricean factor K (dB),
    # by numerical integration of the Ricean envelope pdf. Same reference as rayleigh_fade_prob,
    # so K -> -inf reduces to it. A dominant path (higher K) => FEWER deep fades than Rayleigh.
    # NOTE: the book's Ex 8.9 (0.018) used Mathcad with a different (unstated) fade reference and
    # does not reproduce cleanly; this value is internally consistent instead.
    Klin = 10**(K_dB/10.0); sig2 = 1.0
    A2 = 2*sig2*Klin; A = math.sqrt(A2)
    R = math.sqrt(sig2*10**(-fade_db/10.0))
    r = np.linspace(0, R, n)
    pdf = (r/sig2)*np.exp(-(r**2 + A2)/(2*sig2))*np.i0(A*r/sig2)
    return np.sum((pdf[:-1]+pdf[1:])/2*np.diff(r))   # trapezoid

print(f"Ex 8.8: Rayleigh P(>=12 dB fade) = {rayleigh_fade_prob(12):.3f} (book 0.031)")
print(f"Ex 8.9: Ricean P(>=12 dB fade), K=3 dB = {ricean_fade_prob(12, 3.0):.4f}  "
      f"(< Rayleigh's {rayleigh_fade_prob(12):.3f}: a dominant path steadies the signal; "
      f"book's 0.018 used a different reference)")


In [ ]:
def coherence_bandwidth_hz(rms_delay_spread_s, factor=5.0):
    return 1.0/(factor*rms_delay_spread_s)           # eq 8.24 (factor 5..50)

def delay_from_path_m(excess_path_m):
    return excess_path_m/C

def max_symbol_rate_from_delay(delay_s, fraction=0.1):
    return fraction/delay_s                          # delay <= fraction * T_sym

def doppler_shift_hz(velocity_ms, wavelength_m):
    return velocity_ms/wavelength_m                  # eq 8.26

def coherence_time_s(fm_hz, rms=False):
    return (0.423 if rms else 1.0)/fm_hz             # eq 8.27 / 8.25

# Example 8.6: 50 ksym/s -> B ~ 50 kHz = Bc = 1/(5 sigma_tau)
print(f"Ex 8.6: for Bc=50 kHz, rms delay spread = {1/(5*50e3)*1e6:.0f} us (book <= 4 us)")
# Example 8.7: direct 200 m, multireflection 235.2 m -> excess 35.2 m
tau = delay_from_path_m(35.2)
print(f"Ex 8.7: excess 35.2 m -> delay {tau*1e9:.0f} ns, max symbol rate {max_symbol_rate_from_delay(tau)/1e3:.0f} ksps (book 117 ns, 852)")
# Doppler: 30 m/s (108 km/h) at 2.4 GHz
print(f"Doppler @2.4 GHz, 30 m/s: fm={doppler_shift_hz(30, wavelength(2.4e9)):.0f} Hz, Tc={coherence_time_s(doppler_shift_hz(30, wavelength(2.4e9)))*1e3:.1f} ms")


## 17. Indoor Propagation — Ch 9  *(the deployment use case)*

The two site-general models that **are the engine's path-loss layer**. Both **verified below**
against Ex 9.1 & 9.2 (seeded `itu_indoor_pl_db` = eq 9.1, `log_distance_pl_db` = eq 9.2 — my first
blind pass, now confirmed). Indoors, deterministic models are rare (layout/materials/people change),
so statistical models fit to data are the norm.

- **ITU indoor** `PL = 20log f + N·log d + Lf(n) − 28` (eq 9.1) — a modified power law; **N = 20 ⇒
  free space**, N = 18 corridor (channeling), N = 40 through walls / around corners. Table 9.1 → N by
  band+environment; Table 9.2 → floor loss Lf(n). → `ITU_N`, `itu_floor_loss_db()`.
- **Log-distance** `PL = PL(d0) + N·log(d/d0) + Xσ` (eq 9.2) — exponent = N/10; Xσ ~ N(0,σ) shadowing
  (Ch 8). Table 9.4 → N & σ by building. *Rappaport: indoor σ ≈ 13 dB ⇒ ±26 dB (2σ) is normal.*
- **Delay spread:** indoor impulse response `h(t) = e^(−t/S)`, 0<t<tmax (S = rms delay spread) — the
  exponential profile Ch 8.4 pointed to. Table 9.3: S ~ 20–500 ns. → `indoor_impulse_response()`.

*These N / Lf / σ tables are the literal per-environment inputs for the engine's path-loss layer
(the empirical counterpart to the physics Fresnel/absorption of Ch 2).*


In [ ]:
# Table 9.1: ITU distance power-loss coefficient N  (band -> environment -> N)
ITU_N = {
    "900MHz":    {"office": 33, "commercial": 20},
    "1.2-1.3GHz":{"office": 32, "commercial": 22},
    "1.8-2GHz":  {"residential": 28, "office": 30, "commercial": 22},
    "4GHz":      {"office": 28, "commercial": 22},
    "5.2GHz":    {"office": 31},
    "60GHz":     {"office": 22, "commercial": 17},   # assumed same room
}
# N rules of thumb: 20 = free space / open area, 18 = corridor (channeling), 40 = through walls / corners

def itu_floor_loss_db(n, band="1.8-2GHz", env="office"):   # Table 9.2, Lf(n)
    if band == "1.8-2GHz":
        return {"residential": 4*n, "office": 15 + 4*(n-1), "commercial": 6 + 3*(n-1)}[env]
    if band == "900MHz" and env == "office":  return {1: 9, 2: 19, 3: 24}[n]
    if band == "5.2GHz" and env == "office":  return 16      # n = 1 only
    raise ValueError("no ITU floor-loss data for that band/env")

# Example 9.1: 5.2 GHz office, 100 m, N=31
pl_same  = itu_indoor_pl_db(100, 5200, N=31, Lf=0)
pl_floor = itu_indoor_pl_db(100, 5200, N=31, Lf=itu_floor_loss_db(1, "5.2GHz"))
print(f"Ex 9.1: same-floor PL = {pl_same:.0f} dB (book 108), +1 floor = {pl_floor:.0f} dB "
      f"(+{itu_floor_loss_db(1,'5.2GHz')} dB floor loss)")


In [ ]:
# Table 9.4: log-distance N (dB/decade) and shadowing sigma (dB)
LOGDIST_PARAMS = {   # building: (freq_MHz, N, sigma_dB)
    "retail":           (914, 22, 8.7),  "grocery":         (914, 18, 5.2),
    "office_hard_part": (1500, 30, 7.0), "office_soft_900": (900, 24, 9.6),
    "office_soft_1900": (1900, 26, 14.1),"textile_1300":    (1300, 20, 3.0),
    "paper_cereals":    (1300, 18, 6.0), "metalworking":    (1300, 16, 5.8),
}

# Example 9.2: 1.5 GHz office (hard partition), 100 m, 95% coverage -> N=30, sigma=7.0
pl_d0  = fspl_db(1, 1.5e9)                       # reference free-space loss at 1 m
Xs     = shadowing_margin_db(7.0, 0.95)          # z*sigma, z(95%) = 1.645
median = log_distance_pl_db(100, n=30/10, pl_d0_db=pl_d0)   # exponent = N/10 = 3.0
print(f"Ex 9.2: PL(1m)={pl_d0:.0f} dB, Xs(95%)={Xs:.1f} dB (book 11.5), "
      f"total={median+Xs:.1f} dB (book 107.5); FSL@100m={fspl_db(100,1.5e9):.0f} dB (book 76)")


In [ ]:
# Table 9.3: rms delay spread S (ns) -- often / median / rarely
INDOOR_DELAY_SPREAD_NS = {
    "1.9GHz_residential": (20, 70, 150), "1.9GHz_office":     (35, 100, 460),
    "1.9GHz_commercial":  (55, 150, 500), "5.2GHz_office":    (45, 75, 150),
}
def indoor_impulse_response(t_s, S_s, tmax_s):
    # ITU indoor channel impulse response: h(t) = exp(-t/S) for 0 < t < tmax, else 0.
    t = np.asarray(t_s, float)
    return np.where((t > 0) & (t < tmax_s), np.exp(-t/S_s), 0.0)

t = np.linspace(-0.5e-6, 3e-6, 400)
plt.figure(figsize=(6,3.5))
plt.plot(t*1e6, indoor_impulse_response(t, 1e-6, 2.5e-6))
plt.xlabel("t (us)"); plt.ylabel("h(t)"); plt.title("ITU indoor impulse response (S=1 us, tmax=2.5 us)")
plt.grid(True); plt.show()
# a median office delay spread -> coherence bandwidth (Ch 8.4)
S = 100e-9
print(f"median 1.9 GHz office S=100 ns -> coherence BW ~ {coherence_bandwidth_hz(S)/1e6:.1f} MHz "
      f"(a 20 MHz Wi-Fi channel is wider -> frequency-selective, needs OFDM)")


## 18. Chapter 9 Exercises — worked with the encoded functions

Solving the end-of-chapter problems with the notebook's own functions — a working-tool check that
also exercises §16–§17. Where the book under-specifies a parameter, the assumption is stated.


In [ ]:
# Ex 9-1: median ITU-indoor PL, 1.9 GHz office, 100 m  (N=30 office, Table 9.1; Lf=0 same floor)
pl1 = itu_indoor_pl_db(100, 1900, N=30, Lf=0)
print(f"Ex 9-1: ITU median PL = {pl1:.1f} dB")

# Ex 9-2: PL vs probability of occurrence. ITU gives no sigma, so borrow a typical office sigma = 8 dB.
sigma = 8.0
p = np.linspace(0.01, 0.99, 200)
plt.figure(figsize=(6,4))
plt.plot(p*100, pl1 + norm_ppf(p)*sigma)
plt.xlabel("probability that PL <= value  (%)"); plt.ylabel("path loss (dB)")
plt.title(f"Ex 9-2: ITU median {pl1:.0f} dB, log-normal sigma={sigma} dB"); plt.grid(True); plt.show()


In [ ]:
# Ex 9-3: log-distance, 1.9 GHz office SOFT partition, 98% coverage at 100 m (Table 9.4: N=26, sigma=14.1)
N3, sig3 = 26, 14.1
pl_d0_3  = fspl_db(1, 1.9e9)
median3  = log_distance_pl_db(100, n=N3/10, pl_d0_db=pl_d0_3)
Xs3      = shadowing_margin_db(sig3, 0.98)
print(f"Ex 9-3: PL(1m)={pl_d0_3:.1f}, median={median3:.1f}, Xs(98%)={Xs3:.1f} -> total = {median3+Xs3:.1f} dB")

# Ex 9-4: log-distance, 900 MHz office HARD partition, 38 m, min/max at 99% probability.
# Table 9.4's hard-partition entry is 1500 MHz (N=30, sigma=7.0); used here for the 900 MHz office-hard case.
N4, sig4  = 30, 7.0
pl_d0_4   = fspl_db(1, 900e6)
median4   = log_distance_pl_db(38, n=N4/10, pl_d0_db=pl_d0_4)
half      = norm_ppf(0.995)*sig4                      # central 99% -> +/- z(0.995)*sigma
print(f"Ex 9-4: median={median4:.1f} dB; central-99% range = [{median4-half:.1f}, {median4+half:.1f}] dB (+/-{half:.1f})")


In [ ]:
# Ex 9-5: highest symbol rate with NO equalizer (channel must be flat) from Table 9.3 delay spreads.
# No equalizer -> signal BW <= coherence BW (equivalently symbol period >> delay spread).
# Use the LARGEST tabulated rms delay spread (worst case) to be safe: commercial 1.9 GHz "rarely" = 500 ns.
S_worst  = 500e-9
Bc       = coherence_bandwidth_hz(S_worst)           # 1/(5 S)
Rs_10pct = 0.1/S_worst                               # symbol >= 10x delay spread
print(f"Ex 9-5: worst-case S={S_worst*1e9:.0f} ns -> Bc={Bc/1e3:.0f} kHz (Rs <= {Bc/1e3:.0f} ksps), "
      f"or 10%-of-symbol rule Rs <= {Rs_10pct/1e3:.0f} ksps")
print("        Reasoning: without equalization the channel must appear flat, so the symbol period must be")
print("        >> the delay spread; using the largest tabulated S gives the safe (conservative) rate.")


## 19. Rain & Fog Attenuation — Ch 10  *(future, outdoor mmWave)*

Skip for indoor; the dominant availability limiter for **outdoor links above ~10 GHz**. Rain fade is
`γ·d` scaled by a nonlinear distance factor. Verified below against Ex 10.1 (ITU) & 10.4 (fog).

- **Specific attenuation** `γ = k·RR^α` dB/km (§10.3.1); k, α are frequency + polarization dependent
  (Table 10.1). Polarization combine (eqs 10.3-10.4): τ = 0° H, 45° circular, 90° V. Horizontal rains
  worse than vertical (elongated drops). → `rain_coeffs()`, `rain_coeff_interp()`, `rain_specific_attenuation()`.
- **ITU model** (eqs 10.5-10.7): `A_0.01 = γ·d·r`, distance factor `r = 1/(1+d/d0)`, effective length
  `d0 = 35·e^(−0.015 RR)`. Availability scaling (eqs 10.8/10.9, by latitude). → `itu_rain_attenuation_db()`,
  `itu_availability_adjust()`. *Verified Ex 10.1: Florida (region N, 95 mm/h), 38.6 GHz, 1.1 km →
  23.8 dB @0.01%, 34.3 dB @0.001%.* Valid to 40 GHz / 60 km.
- **Fog/cloud** `γc = Kl·M` dB/km (eq 10.15). → `fog_attenuation_db()`. *Verified Ex 10.4: 30 GHz, heavy
  fog → 4.7 dB over 15 km.*
- **Crane global model:** two-segment, valid ≤22.5 km, same k/α but different rain regions. ⚠ Its
  eqs 10.11/10.13 are garbled in this OCR (couldn't reproduce Ex 10.2's y=−0.189/43.9 dB), so only the
  distance breakpoint `d(RR)` (eq 10.12) and z (eq 10.14, verified) are given — use Crane 1996/2003 for the full model.


In [ ]:
# Table 10.1: rain regression coefficients (freq_GHz -> kH, kV, aH, aV); used by ITU & Crane
RAIN_COEFFS = {
    2:(6.5e-4, 5.91e-4, 1.121, 1.075), 6:(1.75e-3, 1.55e-3, 1.308, 1.265),
    8:(4.54e-3, 3.95e-3, 1.327, 1.310), 10:(0.0101, 0.00887, 1.276, 1.264),
    12:(0.0188, 0.0168, 1.217, 1.200), 20:(0.0751, 0.0691, 1.099, 1.065),
    30:(0.187, 0.167, 1.021, 1.000), 40:(0.350, 0.310, 0.939, 0.929),
}
def rain_coeff_interp(f_ghz):
    # Interpolate (kH,kV,aH,aV): k on a log-log scale, alpha linear vs log(f) (ITU rule).
    fs = sorted(RAIN_COEFFS); lf = np.log10(f_ghz); lfs = np.log10(fs)
    kH = 10**np.interp(lf, lfs, [np.log10(RAIN_COEFFS[f][0]) for f in fs])
    kV = 10**np.interp(lf, lfs, [np.log10(RAIN_COEFFS[f][1]) for f in fs])
    aH = np.interp(lf, lfs, [RAIN_COEFFS[f][2] for f in fs])
    aV = np.interp(lf, lfs, [RAIN_COEFFS[f][3] for f in fs])
    return kH, kV, aH, aV

def rain_coeffs(k_H, k_V, a_H, a_V, tau_deg, theta_deg=0.0):
    # Polarization/elevation-combined k, alpha (eqs 10.3-10.4). tau: 0=H, 45=circular, 90=V.
    ct = np.cos(np.radians(theta_deg))**2; c2t = np.cos(np.radians(2*tau_deg))
    k = (k_H + k_V + (k_H - k_V)*ct*c2t)/2
    a = (k_H*a_H + k_V*a_V + (k_H*a_H - k_V*a_V)*ct*c2t)/(2*k)
    return k, a

def rain_specific_attenuation(RR_mm_h, k, a):
    return k*RR_mm_h**a                                    # dB/km

kH, kV, aH, aV = rain_coeff_interp(38.6)                   # Ex 10.1 uses horizontal (tau=0)
kh, ah = rain_coeffs(kH, kV, aH, aV, tau_deg=0)
print(f"38.6 GHz horizontal: k={kh:.3f}, alpha={ah:.2f} (book 0.324, 0.95)")


In [ ]:
ITU_RAIN_RATE_001 = {   # Table 10.2: 0.01% rain rate (mm/h) by ITU region
    "A":8, "B":12, "C":15, "D":19, "E":22, "F":28, "G":30,
    "H":32, "J":35, "K":42, "L":60, "M":63, "N":95, "P":145,
}
def itu_rain_attenuation_db(RR_001, k, a, d_km):
    # ITU 0.01% (99.99%) rain fade, eqs 10.5-10.7.
    gamma = rain_specific_attenuation(RR_001, k, a)
    d0 = 35*np.exp(-0.015*min(RR_001, 100))               # eq 10.7 (RR capped at 100 mm/h)
    r  = 1/(1 + d_km/d0)                                  # eq 10.6 distance factor
    return gamma*d_km*r

def itu_availability_adjust(atten_001, availability_pct, low_latitude=False):
    # Scale the 0.01% fade to another availability (eqs 10.8/10.9). low_latitude: |lat| < 30 deg.
    p = 100 - availability_pct                            # outage percentage
    if low_latitude:
        return atten_001*0.07*p**(-(0.855 + 0.139*np.log10(p)))   # eq 10.9
    return atten_001*0.12*p**(-(0.546 + 0.043*np.log10(p)))       # eq 10.8

# Example 10.1: Florida (region N, 95 mm/h), 38.6 GHz horizontal, 1.1 km
a001 = itu_rain_attenuation_db(ITU_RAIN_RATE_001["N"], kh, ah, 1.1)
a5   = itu_availability_adjust(a001, 99.999, low_latitude=True)
print(f"Ex 10.1: A(0.01%)={a001:.1f} dB (book 23.8), A(0.001%, five-nines)={a5:.1f} dB (book 34.3)")


In [ ]:
def fog_attenuation_db(Kl, M_g_m3, d_km):
    return Kl*M_g_m3*d_km                                 # eq 10.15, gamma_c = Kl*M

def crane_breakpoint_km(RR_mm_h):
    return 3.8 - 0.6*np.log(RR_mm_h)                      # eq 10.12, d(RR)  (Crane, partial)

# Example 10.4: 30 GHz, 10 C, heavy fog (M=0.5), 15 km; Kl(30 GHz,10 C)=0.63 (dB/km)/(g/m3)
print(f"Ex 10.4: fog gamma = {fog_attenuation_db(0.63, 0.5, 1):.3f} dB/km -> "
      f"{fog_attenuation_db(0.63, 0.5, 15):.1f} dB over 15 km (book 4.7)")
# Crane breakpoint + z for Ex 10.2 (RR=176): z verified, y/attenuation not (OCR garbled)
rr = 176
print(f"Crane d(RR={rr}) = {crane_breakpoint_km(rr):.3f} km, z = {0.95*(0.026 - 0.03*np.log(rr)):.5f} "
      f"(book z=-0.12266; full Crane atten not reproduced)")


## 20. Satellite Links — Ch 11  *(future, outdoor)*

The final chapter, and a capstone that **reuses everything**: FSPL (§1), the rain model (§19), and
the noise floor (§10). Skip for the indoor engine (single long slant path through the whole
atmosphere), but the geometry and hot-pad noise are clean and worth having.

- **Slant-range geometry:** central angle ψ from lat/long (eq 11.4), slant range `rs` (eq 11.5),
  elevation `θ` (eq 11.6). GEO radius ≈ 42,242 km from earth center. → `sat_central_angle()`,
  `sat_slant_range_km()`, `sat_elevation_deg()`. *Verified Ex 11.1 (rs=36,314 km, θ=66.6°).*
- **ITU satellite rain** (P.618 10-step, eqs 11.9-11.25) extends §19 with a rain-cell-height
  geometry. → `itu_sat_rain_atten_db()`. *Verified Ex 11.3 uplink (A0.01=38.5 dB, A0.1=14.8 dB).*
- **Noise (absorptive losses raise the noise floor):** hot-pad `TN = (Tin + (L−1)T)/L` (eq 11.50),
  rain temp `Tr = Tm(1 − 10^(−A/10))` (eq 11.51), figure of merit `G/T`. → `hotpad_noise_temp()`,
  `rain_noise_temp()`, `gt_db()`. *Verified Ex 11.5/11.6 (TN=222.5 K, Tr=204.4 K).*
- **Doppler** `fd = vr·f0/c` (eq 11.1) = §16 `doppler_shift_hz`. GEO round-trip delay ≈ 240 ms.
  Ionospheric effects (Faraday rotation, scintillation) only matter <10 GHz → circular pol preferred.


In [ ]:
def sat_central_angle(Le, le, Ls, ls):
    # Central angle psi (deg) from earth-station & subsatellite lat/long (deg). Eq 11.4.
    Le, le, Ls, ls = map(np.radians, (Le, le, Ls, ls))
    return np.degrees(np.arccos(np.cos(Le)*np.cos(Ls)*np.cos(ls-le) + np.sin(Le)*np.sin(Ls)))

def sat_slant_range_km(psi_deg, h_km=42242.0, re_km=6378.0):
    p = np.radians(psi_deg)
    return h_km*np.sqrt(1 + (re_km/h_km)**2 - 2*(re_km/h_km)*np.cos(p))     # eq 11.5

def sat_elevation_deg(psi_deg, rs_km, h_km=42242.0):
    return np.degrees(np.arccos(h_km*np.sin(np.radians(psi_deg))/rs_km))    # eq 11.6

# Example 11.1: GEO sat, earth station at 20 deg latitude, same longitude
rs = sat_slant_range_km(20)
print(f"Ex 11.1: slant range={rs:.0f} km (book 36,314), elevation={sat_elevation_deg(20, rs):.1f} deg (book 66.6)")
print(f"         FSPL to that GEO sat @12 GHz = {fspl_db(rs*1e3, 12e9):.1f} dB")


In [ ]:
def itu_sat_rain_atten_db(freq_ghz, elev_deg, lat_deg, RR001, k, a, hs_km=0.0, availability_pct=99.99):
    # ITU-R P.618 slant-path rain fade (10-step), returns fade depth (dB) at the given availability.
    th = elev_deg; st = np.sin(np.radians(th))
    hR = 4.0 if abs(lat_deg) < 36 else 4.0 - 0.075*(abs(lat_deg) - 36)      # step 1
    if th < 5:                                                             # step 2
        Lsl = 2*(hR-hs_km)/(np.sqrt(st**2 + 2*(hR-hs_km)/8500) + st)
    else:
        Lsl = (hR - hs_km)/st
    LG = Lsl*np.cos(np.radians(th))                                        # step 3
    gR = k*RR001**a                                                       # step 5
    r001 = 1/(1 + 0.78*np.sqrt(LG*gR/freq_ghz) - 0.38*(1 - np.exp(-2*LG))) # step 6
    zeta = np.degrees(np.arctan((hR - hs_km)/(LG*r001)))                   # step 7
    LR = LG*r001/np.cos(np.radians(th)) if zeta > th else (hR - hs_km)/st
    chi = 36 - abs(lat_deg) if abs(lat_deg) < 36 else 0.0
    v001 = 1/(1 + np.sqrt(st)*(31*(1 - np.exp(-th/(1+chi)))*np.sqrt(LR*gR)/freq_ghz**2 - 0.45))
    A001 = gR*(LR*v001)                                                    # steps 8-9
    p = 100 - availability_pct                                            # step 10
    if p >= 1 or abs(lat_deg) >= 36:      b = 0.0
    elif th > 25:                          b = -0.005*(abs(lat_deg) - 36)
    else:                                  b = -0.005*(abs(lat_deg) - 36) + 1.8 - 4.25*st
    exp = -(0.655 + 0.033*np.log(p) - 0.045*np.log(A001) - b*(1-p)*st)
    return A001*(p/0.01)**exp

# Example 11.3 uplink: 30 GHz, NYC (region K, 42 mm/h, lat 40), circular -> k=0.177, a=1.011, elev 40.9
A001 = itu_sat_rain_atten_db(30, 40.9, 40, 42, 0.177, 1.011, availability_pct=99.99)
A01  = itu_sat_rain_atten_db(30, 40.9, 40, 42, 0.177, 1.011, availability_pct=99.9)
print(f"Ex 11.3 uplink: A(0.01%)={A001:.1f} dB (book 38.5), A(0.1%)={A01:.1f} dB (book 14.8)")


In [ ]:
def hotpad_noise_temp(Tin_K, L_db, T_atten_K=290.0):
    # Attenuator (loss L) raises the noise temp: TN = (Tin + (L-1) T)/L. Eq 11.50.
    L = 10**(L_db/10.0)
    return (Tin_K + (L - 1)*T_atten_K)/L

def rain_noise_temp(A_db, Tm_K=273.0):
    return Tm_K*(1 - 10**(-A_db/10.0))                    # eq 11.51 (added to Tsys)

def gt_db(G_db, Tsys_K):
    return G_db - 10*np.log10(Tsys_K)                     # figure of merit G/T (dB/K)

# Example 11.5 (hot-pad) & 11.6 (rain temp): Ta=20 K, Te=50 K, 6 dB rain fade
Ta, Te, fade = 20.0, 50.0, 6.0
TN   = hotpad_noise_temp(Ta, fade)                        # hot-pad
Tsys_rain_hp = TN + Te
Tr   = rain_noise_temp(fade)                              # rain-temperature method
Tsys_rain_rt = Ta + Tr + Te
print(f"Ex 11.5 hot-pad: TN={TN:.1f} K, Tsys_rain={Tsys_rain_hp:.1f} K (book 222.5 / 272.5)")
print(f"Ex 11.6 rain-T : Tr={Tr:.1f} K, Tsys_rain={Tsys_rain_rt:.1f} K (book 204.4 / 274.4)")
print(f"  noise up {10*np.log10(Tsys_rain_hp/(Ta+Te)):.1f} dB + signal down {fade:.0f} dB = "
      f"{10*np.log10(Tsys_rain_hp/(Ta+Te))+fade:.1f} dB SNR loss (book Ex11.5 says 9.9 = typo; Ex11.6 says 11.9)")
print(f"  G/T (G=30 dBi, Tsys=70 K) = {gt_db(30, 70):.1f} dB/K (book prints 14.3 = the linear ratio 1000/70, mislabeled)")


---
## Corpus complete — Ch 1–11 encoded & verified

Every engine-relevant equation from **EE 625 Radio Wave Propagation** is now a runnable, example-
checked function (§0–§20). The three Essentials (Ch 2/8/9) drive the indoor engine; Ch 5-7/10-11 are
the outdoor/future-feature layer. Four blind-seeded functions and every worked example that survived
OCR were verified; book slips were flagged, not silently reproduced. Ch 12 (RF Safety) is the only
remaining chapter and is out of scope for propagation.
